# HLA-B39 Simulation Analysis - Visualization & Analysis

This notebook performs analysis and visualization using preprocessed data.

**Prerequisites:** Run `1_preprocessing.py` first to generate CSV files.

---

In [1]:
import glob
from prody import *
import pandas as pd
import re
import numpy as np
import tqdm
import natsort
import seaborn as sns
import scipy.stats
import matplotlib.pyplot as plt
import py3Dmol
import networkx as nx

## Load Preprocessed Data

Load all CSV files generated by the preprocessing script.

In [2]:
# Initialize common variables and load all preprocessed datasets
BASE_FOLDER = '/mnt/MORIA/experiments/hla_b39/2022_10_14_bioe_leviathan_backup/T1DM_B39_traj_analysis/gmx_traj_data/'
frame_cols = list(map(str, np.arange(0, 239, 1)))

# Load all CSV files from preprocessing
print('Loading preprocessed datasets...')
df_intEn = pd.read_csv('intEnVdW_2025_07_10.csv')
df_cons_pairs = pd.read_csv('df_cons_pairs_saved_2025_07_10.csv')
df_sig_aff_pairs_b39 = pd.read_csv('df_sig_aff_pairs_b39_2025_07_10.csv')
df_sig_aff_pairs_loaded = pd.read_csv('df_sig_aff_pairs_loaded_2025_07_10.csv')
df_bc_equil = pd.read_csv('df_bc_equil_2025_07_10.csv')
df_bc_equil_cons_resids = pd.read_csv('df_bc_equil_cons_resids_2025_07_10.csv')
df_bc_sig_aff_resids = pd.read_csv('df_bc_sigaff_resids_2026_01_27.csv')

print('All datasets loaded successfully.')
print(f'  - df_intEn: {len(df_intEn)} rows')
print(f'  - df_cons_pairs: {len(df_cons_pairs)} rows')
print(f'  - df_sig_aff_pairs_b39: {len(df_sig_aff_pairs_b39)} rows')
print(f'  - df_sig_aff_pairs_loaded: {len(df_sig_aff_pairs_loaded)} rows')
print(f'  - df_bc_equil_cons_resids: {len(df_bc_equil_cons_resids)} rows')
print(f'  - df_bc_sig_aff_resids: {len(df_bc_sig_aff_resids)} rows')

Loading preprocessed datasets...


/tmp/ipykernel_2717707/3034445152.py:7: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df_intEn = pd.read_csv('intEnVdW_2025_07_10.csv')
/tmp/ipykernel_2717707/3034445152.py:8: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df_cons_pairs = pd.read_csv('df_cons_pairs_saved_2025_07_10.csv')


EmptyDataError: No columns to parse from file

## 2.2 Summary Statistics

Overview of interaction changes by allele and peptide across systems.

In [ ]:
# Get unique alleles and peptides from the data
unique_alleles = df_sig_aff_pairs_loaded['allele'].unique()
unique_peptides = df_sig_aff_pairs_loaded['peptide'].unique()

print("=== INTERACTION CHANGES BY ALLELE AND PEPTIDE ===\n")

# Create a mapping of peptides to their full names (you can expand this)
peptide_names = {
    'ALL': 'peptide ALL',
    'A': 'peptide A',  # Update with actual peptide name if known
    'AL': 'peptide AL',  # Update with actual peptide name if known
    # Add more mappings as needed
}

for allele in sorted(unique_alleles):
    print(f"--- {allele} ---")
    allele_data = df_sig_aff_pairs_loaded[df_sig_aff_pairs_loaded['allele'] == allele]
    
    for peptide in sorted(allele_data['peptide'].unique()):
        peptide_data = allele_data[allele_data['peptide'] == peptide]
        
        # Get full peptide name if available
        full_peptide_name = peptide_names.get(peptide, peptide)
        
        repulsive_count = len(peptide_data[peptide_data['change_type'] == 'repulsive'])
        attractive_count = len(peptide_data[peptide_data['change_type'] == 'attractive'])
        
        print(f"  Peptide {peptide} ({full_peptide_name}):")
        print(f"    Repulsive interactions: {repulsive_count}")
        print(f"    Attractive interactions: {attractive_count}")
        print(f"    Total interactions: {repulsive_count + attractive_count}")
    
    print()

# Summary statistics
print("=== OVERALL SUMMARY ===")
total_repulsive = len(df_sig_aff_pairs_loaded[df_sig_aff_pairs_loaded['change_type'] == 'repulsive'])
total_attractive = len(df_sig_aff_pairs_loaded[df_sig_aff_pairs_loaded['change_type'] == 'attractive'])
total_interactions = len(df_sig_aff_pairs_loaded)

print(f"Total significantly affected pairs: {total_interactions}")
print(f"Total repulsive changes: {total_repulsive}")
print(f"Total attractive changes: {total_attractive}")

# Breakdown by allele
print(f"\nBreakdown by allele:")
allele_summary = df_sig_aff_pairs_loaded.groupby(['allele', 'change_type']).size().unstack(fill_value=0)
print(allele_summary)

# Breakdown by peptide
print(f"\nBreakdown by peptide:")
peptide_summary = df_sig_aff_pairs_loaded.groupby(['peptide', 'change_type']).size().unstack(fill_value=0)
print(peptide_summary)

In [ ]:
# Explore available data combinations
print("Available alleles:", sorted(df_sig_aff_pairs_loaded['allele'].unique()))
print("Available peptides:", sorted(df_sig_aff_pairs_loaded['peptide'].unique()))
print("\nData combinations (allele, peptide):")
combinations = df_sig_aff_pairs_loaded[['allele', 'peptide']].drop_duplicates().sort_values(['allele', 'peptide'])
for _, row in combinations.iterrows():
    count = len(df_sig_aff_pairs_loaded[(df_sig_aff_pairs_loaded['allele'] == row['allele']) & 
                                       (df_sig_aff_pairs_loaded['peptide'] == row['peptide'])])
    print(f"  {row['allele']} + {row['peptide']}: {count} pairs")

## 2.3 3D Structure Visualization

Visualize significantly affected residue pairs in 3D (red = attractive, blue = repulsive).

In [ ]:
def vis_pairs(pdb_file, df_tovis, pair_color='red'):
    pdb_str = open(pdb_file, 'r').read()
    # Initialize viewer
    view = py3Dmol.view(width=800, height=600)

    # Add model from the PDB string
    view.addModel(pdb_str, 'pdb')

    # Apply cartoon representation for the entire protein
    view.setStyle({'cartoon': {}})

    for pair in df_tovis['pair'].unique():
        res1 = pair.split('-')[0]
        res2 = pair.split('-')[1]
        # Highlight residues 182-210 in chain A with Van der Waals (VDW) representation
        view.setStyle({'chain':res1[-1], 'resi':res1[:-1]}, {'sphere': {'radius': 1.0, 'color': pair_color}})
        view.setStyle({'chain':res2[-1], 'resi':res2[:-1]}, {'sphere': {'radius': 1.0, 'color': pair_color}})

    # Zoom to the selected residues
    view.zoomTo()

    # Show the viewer
    view.show()

### B3801 Visualizations

In [ ]:
# Load your PDB structure as a string
vis_pairs('/mnt/MORIA/experiments/hla_b39/2022_10_14_bioe_leviathan_backup/T1DM_B39_traj_analysis/gmx_traj_data/B3801_ALL_1/grinn_output_skip10/system_dry.pdb',
          df_sig_aff_pairs_loaded.query("allele == 'B3801' and peptide == 'ALL' and change_type == 'attractive'").copy(),
          'red')

In [ ]:
df_sig_aff_pairs_loaded.query("allele == 'B3801' and peptide == 'ALL' and change_type == 'attractive'")

In [ ]:
# Load your PDB structure as a string
vis_pairs('/mnt/MORIA/experiments/hla_b39/2022_10_14_bioe_leviathan_backup/T1DM_B39_traj_analysis/gmx_traj_data/B3801_ALL_1/grinn_output_skip10/system_dry.pdb',
          df_sig_aff_pairs_loaded.query("allele == 'B3801' and peptide == 'ALL' and change_type == 'repulsive'").copy(),
          'blue')

In [ ]:
df_sig_aff_pairs_loaded.query("allele == 'B3801' and peptide == 'ALL' and change_type == 'repulsive'")

In [ ]:
# Load your PDB structure as a string
vis_pairs('/mnt/MORIA/experiments/hla_b39/2022_10_14_bioe_leviathan_backup/T1DM_B39_traj_analysis/gmx_traj_data/B3801_AL_1/grinn_output_skip10/system_dry.pdb',
          df_sig_aff_pairs_loaded.query("allele == 'B3801' and peptide == 'AL' and change_type == 'attractive'").copy(),
          'red')

In [ ]:
df_sig_aff_pairs_loaded.query("allele == 'B3801' and peptide == 'AL' and change_type == 'attractive'")

In [ ]:
# Load your PDB structure as a string
vis_pairs('/mnt/MORIA/experiments/hla_b39/2022_10_14_bioe_leviathan_backup/T1DM_B39_traj_analysis/gmx_traj_data/B3801_AL_1/grinn_output_skip10/system_dry.pdb',
          df_sig_aff_pairs_loaded.query("allele == 'B3801' and peptide == 'AL' and change_type == 'repulsive'").copy(),
          'blue')

In [ ]:
df_sig_aff_pairs_loaded.query("allele == 'B3801' and peptide == 'AL' and change_type == 'repulsive'")

In [ ]:
# Load your PDB structure as a string
vis_pairs('/mnt/MORIA/experiments/hla_b39/2022_10_14_bioe_leviathan_backup/T1DM_B39_traj_analysis/gmx_traj_data/B3801_AL_1/grinn_output_skip10/system_dry.pdb',
          df_sig_aff_pairs_loaded.query("allele == 'B3801' and peptide == 'A' and change_type == 'attractive'").copy(),
          'red')

In [ ]:
df_sig_aff_pairs_loaded.query("allele == 'B3801' and peptide == 'A' and change_type == 'attractive'")

In [ ]:
# Load your PDB structure as a string
vis_pairs('/mnt/MORIA/experiments/hla_b39/2022_10_14_bioe_leviathan_backup/T1DM_B39_traj_analysis/gmx_traj_data/B3801_AL_1/grinn_output_skip10/system_dry.pdb',
          df_sig_aff_pairs_loaded.query("allele == 'B3801' and peptide == 'A' and change_type == 'repulsive'").copy(),
          'blue')

In [ ]:
df_sig_aff_pairs_loaded.query("allele == 'B3801' and peptide == 'A' and change_type == 'repulsive'")

### B3901 Visualizations

In [ ]:
# B3901 ALL - Attractive interactions
vis_pairs('/mnt/MORIA/experiments/hla_b39/2022_10_14_bioe_leviathan_backup/T1DM_B39_traj_analysis/gmx_traj_data/B3901_ALL_1/grinn_output_skip10/system_dry.pdb',
          df_sig_aff_pairs_loaded.query("allele == 'B3901' and peptide == 'ALL' and change_type == 'attractive'").copy(),
          'red')

In [ ]:
df_sig_aff_pairs_loaded.query("allele == 'B3901' and peptide == 'ALL' and change_type == 'attractive'")

In [ ]:
# B3901 ALL - Repulsive interactions
vis_pairs('/mnt/MORIA/experiments/hla_b39/2022_10_14_bioe_leviathan_backup/T1DM_B39_traj_analysis/gmx_traj_data/B3901_ALL_1/grinn_output_skip10/system_dry.pdb',
          df_sig_aff_pairs_loaded.query("allele == 'B3901' and peptide == 'ALL' and change_type == 'repulsive'").copy(),
          'blue')

In [ ]:
df_sig_aff_pairs_loaded.query("allele == 'B3901' and peptide == 'ALL' and change_type == 'repulsive'")

In [ ]:
# B3901 AL - Attractive interactions
vis_pairs('/mnt/MORIA/experiments/hla_b39/2022_10_14_bioe_leviathan_backup/T1DM_B39_traj_analysis/gmx_traj_data/B3901_AL_1/grinn_output_skip10/system_dry.pdb',
          df_sig_aff_pairs_loaded.query("allele == 'B3901' and peptide == 'AL' and change_type == 'attractive'").copy(),
          'red')

In [ ]:
df_sig_aff_pairs_loaded.query("allele == 'B3901' and peptide == 'AL' and change_type == 'attractive'")

In [ ]:
# B3901 AL - Repulsive interactions
vis_pairs('/mnt/MORIA/experiments/hla_b39/2022_10_14_bioe_leviathan_backup/T1DM_B39_traj_analysis/gmx_traj_data/B3901_AL_1/grinn_output_skip10/system_dry.pdb',
          df_sig_aff_pairs_loaded.query("allele == 'B3901' and peptide == 'AL' and change_type == 'repulsive'").copy(),
          'blue')

In [ ]:
df_sig_aff_pairs_loaded.query("allele == 'B3901' and peptide == 'AL' and change_type == 'repulsive'")

In [ ]:
# B3901 A - Attractive interactions
vis_pairs('/mnt/MORIA/experiments/hla_b39/2022_10_14_bioe_leviathan_backup/T1DM_B39_traj_analysis/gmx_traj_data/B3901_A_1/grinn_output_skip10/system_dry.pdb',
          df_sig_aff_pairs_loaded.query("allele == 'B3901' and peptide == 'A' and change_type == 'attractive'").copy(),
          'red')

In [ ]:
df_sig_aff_pairs_loaded.query("allele == 'B3901' and peptide == 'A' and change_type == 'attractive'")

In [ ]:
# B3901 A - Repulsive interactions
vis_pairs('/mnt/MORIA/experiments/hla_b39/2022_10_14_bioe_leviathan_backup/T1DM_B39_traj_analysis/gmx_traj_data/B3901_A_1/grinn_output_skip10/system_dry.pdb',
          df_sig_aff_pairs_loaded.query("allele == 'B3901' and peptide == 'A' and change_type == 'repulsive'").copy(),
          'blue')

In [ ]:
df_sig_aff_pairs_loaded.query("allele == 'B3901' and peptide == 'A' and change_type == 'repulsive'")

### B3906 Visualizations

In [ ]:
# B3906 ALL - Attractive interactions
vis_pairs('/mnt/MORIA/experiments/hla_b39/2022_10_14_bioe_leviathan_backup/T1DM_B39_traj_analysis/gmx_traj_data/B3906_ALL_1/grinn_output_skip10/system_dry.pdb',
          df_sig_aff_pairs_loaded.query("allele == 'B3906' and peptide == 'ALL' and change_type == 'attractive'").copy(),
          'red')

In [ ]:
df_sig_aff_pairs_loaded.query("allele == 'B3906' and peptide == 'ALL' and change_type == 'attractive'")

In [ ]:
# B3906 ALL - Repulsive interactions
vis_pairs('/mnt/MORIA/experiments/hla_b39/2022_10_14_bioe_leviathan_backup/T1DM_B39_traj_analysis/gmx_traj_data/B3906_ALL_1/grinn_output_skip10/system_dry.pdb',
          df_sig_aff_pairs_loaded.query("allele == 'B3906' and peptide == 'ALL' and change_type == 'repulsive'").copy(),
          'blue')

In [ ]:
df_sig_aff_pairs_loaded.query("allele == 'B3906' and peptide == 'ALL' and change_type == 'repulsive'")

In [ ]:
# B3906 AL - Attractive interactions
vis_pairs('/mnt/MORIA/experiments/hla_b39/2022_10_14_bioe_leviathan_backup/T1DM_B39_traj_analysis/gmx_traj_data/B3906_AL_1/grinn_output_skip10/system_dry.pdb',
          df_sig_aff_pairs_loaded.query("allele == 'B3906' and peptide == 'AL' and change_type == 'attractive'").copy(),
          'red')

In [ ]:
df_sig_aff_pairs_loaded.query("allele == 'B3906' and peptide == 'AL' and change_type == 'attractive'")

In [ ]:
# B3906 AL - Repulsive interactions
vis_pairs('/mnt/MORIA/experiments/hla_b39/2022_10_14_bioe_leviathan_backup/T1DM_B39_traj_analysis/gmx_traj_data/B3906_AL_1/grinn_output_skip10/system_dry.pdb',
          df_sig_aff_pairs_loaded.query("allele == 'B3906' and peptide == 'AL' and change_type == 'repulsive'").copy(),
          'blue')

In [ ]:
df_sig_aff_pairs_loaded.query("allele == 'B3906' and peptide == 'AL' and change_type == 'repulsive'")

In [ ]:
# B3906 A - Attractive interactions
vis_pairs('/mnt/MORIA/experiments/hla_b39/2022_10_14_bioe_leviathan_backup/T1DM_B39_traj_analysis/gmx_traj_data/B3906_A_1/grinn_output_skip10/system_dry.pdb',
          df_sig_aff_pairs_loaded.query("allele == 'B3906' and peptide == 'A' and change_type == 'attractive'").copy(),
          'red')

In [ ]:
df_sig_aff_pairs_loaded.query("allele == 'B3906' and peptide == 'A' and change_type == 'attractive'")

In [ ]:
# B3906 A - Repulsive interactions
vis_pairs('/mnt/MORIA/experiments/hla_b39/2022_10_14_bioe_leviathan_backup/T1DM_B39_traj_analysis/gmx_traj_data/B3906_A_1/grinn_output_skip10/system_dry.pdb',
          df_sig_aff_pairs_loaded.query("allele == 'B3906' and peptide == 'A' and change_type == 'repulsive'").copy(),
          'blue')

---
**CONTINUE PREPROCESSING BELOW:** Resume with section 1.6 to complete the preprocessing steps before running further analysis sections.

## 2.4 Conservation and Network Centrality

Analyze the relationship between evolutionary conservation scores and network centrality changes.

In [ ]:
df_bc_sig_aff_resids.sort_values(by='mean_median_bc', ascending=False, inplace=True)

In [ ]:
df_bc_sig_aff_resids

In [ ]:
# Read 5N1Y_With_Conservation_Scores.pdb from the current directory
pdb_file = '5N1Y_With_Conservation_Scores.pdb'
pdb_str = open(pdb_file, 'r').read()

# Start an emptry dictionary to store conservation scores
cons_scores = dict()
# Loop through lines in the PDB file
lines = pdb_str.split('\n')
for line in lines:
    if line.startswith('ATOM') and 'CA' in line:
        line = line.split()
        chain_id = line[4]
        cons_score = line[-2]
        if cons_score.startswith('1.00'):
            cons_score = float(cons_score[4:])
        elif cons_score.startswith('0.50'):
            cons_score = float(cons_score[4:])
        else:
            cons_score = float(cons_score)
        res_num = line[5]
        resnum_chain_id = f'{res_num}_{chain_id}'
        cons_scores[resnum_chain_id] = cons_score

# Create a new column in df_bc_sig_aff_resids for conservation scores
df_bc_sig_aff_resids['cons_score'] = np.nan
# Loop through the DataFrame and assign conservation scores
for i, row in df_bc_sig_aff_resids.iterrows():
    resnum_chain_id = row['Residue_id']
    if resnum_chain_id in cons_scores:
        df_bc_sig_aff_resids.at[i, 'cons_score'] = cons_scores[resnum_chain_id]
    else:
        print(f"Warning: {resnum_chain_id} not found in conservation scores.")
df_bc_sig_aff_resids['cons_score'] = df_bc_sig_aff_resids['cons_score'].astype(float)

### Integrate evolutionary conservation scores
Parse conservation scores from PDB file and correlate with betweenness centrality values.

In [ ]:
df_bc_sig_aff_resids[df_bc_sig_aff_resids['Residue_id'].str.contains('_A')].sort_values(by='cons_score', ascending=False).head(40)

In [ ]:
# Plot mean_median_bc vs cons_score
plt.figure(figsize=(10, 6))
# Increase font size to make it more readable
plt.rcParams.update({'font.size': 14})
plt.scatter(df_bc_sig_aff_resids[df_bc_sig_aff_resids['Residue_id'].str.contains('_A')]['cons_score'], 
            df_bc_sig_aff_resids[df_bc_sig_aff_resids['Residue_id'].str.contains('_A')]['mean_median_bc'], alpha=0.5)
plt.xlabel('Conservation Score')
plt.ylabel('Mean Median BC')
# Annotate the points with their residue IDs, but only for those with mean BC above 0.06
for i, row in df_bc_sig_aff_resids[df_bc_sig_aff_resids['Residue_id'].str.contains('_A')].iterrows():
    if row['mean_median_bc'] > 0.06:
        plt.annotate(row['Residue_id'], (row['cons_score'], row['mean_median_bc']), fontsize=8)
plt.grid()
plt.tight_layout()
plt.title('Mean Median BC vs Conservation Score')

### Visualize conservation vs network centrality relationship

In [ ]:
df_bc_sig_aff_resids.sort_values(by='diff_bc_3utq_pf',ascending=False, inplace=True)
top10_bc_aff_3utq = df_bc_sig_aff_resids[:10]
top10_bc_aff_3utq

### Identify top residues with increased centrality upon peptide binding

In [ ]:
df_bc_sig_aff_resids

In [ ]:
df_bc_sig_aff_resids.sort_values(by='diff_bc_5n1y_pf',ascending=False, inplace=True)
top10_bc_aff_5n1y = df_bc_sig_aff_resids[:10]
top10_bc_aff_5n1y

In [ ]:
df_bc_sig_aff_resids.sort_values(by='diff_bc_5c0f_pf',ascending=False, inplace=True)
top10_bc_aff_5c0f = df_bc_sig_aff_resids[:10]
top10_bc_aff_5c0f

In [ ]:
top10_bc = pd.concat([top10_bc_aff_3utq, top10_bc_aff_5n1y, top10_bc_aff_5c0f],ignore_index=True)
top10_bc.sort_values(by='mean_median_bc',ascending=False,inplace=True)
(fig,ax) = plt.subplots(nrows=1,ncols=1,figsize=(16,3))
df_2plot = df_bc_equil_cons_resids[df_bc_equil_cons_resids['Residue_id'].isin(top10_bc['Residue_id'].values)]
order = top10_bc['Residue_id'].values
sns.boxplot(df_2plot,x='Residue_id',y='BC',hue='peptide', ax= ax, order=order,showfliers=False)

In [ ]:
df_bc_sig_aff_resids.sort_values(by='diff_bc_3utq_pf',ascending=True, inplace=True)
bottom10_bc_aff_3utq = df_bc_sig_aff_resids[:10]
bottom10_bc_aff_3utq

### Identify residues with decreased centrality upon peptide binding

In [ ]:
df_bc_sig_aff_resids.sort_values(by='diff_bc_5c0f_pf',ascending=True, inplace=True)
bottom10_bc_aff_5c0f = df_bc_sig_aff_resids[:10]
bottom10_bc_aff_5c0f

In [ ]:
bottom10_bc = pd.concat([bottom10_bc_aff_3utq, bottom10_bc_aff_5n1y, bottom10_bc_aff_5c0f],ignore_index=True)
bottom10_bc.sort_values(by='mean_median_bc',ascending=False,inplace=True)
(fig,ax) = plt.subplots(nrows=1,ncols=1,figsize=(16,3))
df_2plot = df_bc_equil_cons_resids[df_bc_equil_cons_resids['Residue_id'].isin(bottom10_bc['Residue_id'].values)]
order = bottom10_bc['Residue_id'].values
sns.boxplot(df_2plot,x='Residue_id',y='BC',hue='peptide', ax= ax, order=order,showfliers=False)

## 2.5 Polymorphic Position Analysis

Analyze interaction energies around polymorphic positions.

In [ ]:
def analyze_polymorphic_residue_interactions(df, residue_id, title_suffix="", figsize_per_subplot=(8, 5)):
    # Determine available frame columns dynamically
    potential_frame_cols = [str(i) for i in range(1000)]
    available_frame_cols = [col for col in potential_frame_cols if col in df.columns]
    
    # Get interactions involving the specified residue
    df_residue = df.query(f"resnum_chain1 == '{residue_id}' or resnum_chain2 == '{residue_id}'")
    
    if len(df_residue) == 0:
        print(f"Warning: No interactions found involving residue {residue_id}")
        return {'filtered_df': df_residue, 'unique_pairs': [], 'pair_counts': {}, 'peptide_summary': {}}
    
    print(f"Found {len(df_residue)} interactions involving residue {residue_id}")
    
    # Get unique peptide values and include peptide-free cases
    unique_peptides = df_residue['peptide'].dropna().unique()
    has_peptide_free = df_residue['peptide'].isna().any()
    
    # Create list of conditions to plot
    conditions = []
    if has_peptide_free:
        conditions.append(('Peptide-free', None))
    for peptide in sorted(unique_peptides):
        conditions.append((f'Peptide: {peptide}', peptide))
    
    print(f"Analyzing {len(conditions)} conditions: {len(unique_peptides)} peptide types + {'peptide-free' if has_peptide_free else 'no peptide-free cases'}")
    
    # Get unique interaction pairs
    unique_pairs = df_residue['resnum_chain12'].unique()
    
    # Initialize summary structures
    pair_counts = {}
    peptide_summary = {}
    
    # Create subplots for each interaction pair
    for resnum_chain12 in unique_pairs:
        df_pair = df_residue[df_residue['resnum_chain12'] == resnum_chain12]
        
        if len(df_pair) == 0:
            continue
        
        # Calculate subplot layout
        n_conditions = len(conditions)
        n_cols = 2
        n_rows = 2
        
        # Create figure with subplots
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(figsize_per_subplot[0] * n_cols, figsize_per_subplot[1] * n_rows))
        if n_conditions == 1:
            axes = [axes]
        elif n_rows == 1:
            axes = axes if isinstance(axes, (list, np.ndarray)) else [axes]
        else:
            axes = axes.flatten()
        
        # Plot each condition
        for idx, (condition_name, peptide_value) in enumerate(conditions):
            ax = axes[idx]
            
            # Filter data for this condition
            if peptide_value is None:  # Peptide-free
                df_condition = df_pair[df_pair['peptide'].isna()]
                condition_key = 'peptide_free'
            else:  # Specific peptide
                df_condition = df_pair[df_pair['peptide'] == peptide_value]
                condition_key = peptide_value
            
            if len(df_condition) == 0:
                ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
                ax.set_title(f'{condition_name}\n(No data)')
                continue
            
            # Initialize condition in summaries if not exists
            if condition_key not in peptide_summary:
                peptide_summary[condition_key] = {}
            
            alleles_plotted = []
            
            # Plot KDE for each allele in this condition
            for allele in df_condition['allele'].unique():
                df_allele = df_condition[df_condition['allele'] == allele]
                
                if len(df_allele) == 0:
                    continue
                    
                interaction_energies = df_allele[available_frame_cols].values.flatten()
                interaction_energies = interaction_energies[~np.isnan(interaction_energies)]
                
                if len(interaction_energies) == 0:
                    continue
                    
                sns.kdeplot(interaction_energies, label=allele, fill=True, alpha=0.7, ax=ax)
                alleles_plotted.append(allele)
                
                # Count pairs per allele and condition
                pair_key = f"{condition_key}_{allele}"
                if pair_key not in pair_counts:
                    pair_counts[pair_key] = 0
                pair_counts[pair_key] += 1
                
                # Update peptide summary
                if allele not in peptide_summary[condition_key]:
                    peptide_summary[condition_key][allele] = 0
                peptide_summary[condition_key][allele] += 1
            
            # Customize subplot
            if alleles_plotted:
                ax.set_title(f'{condition_name}\n{resnum_chain12}')
                ax.set_xlabel('Interaction Energy (kJ/mol)')
                ax.set_ylabel('Density')
                ax.legend(fontsize='small')
                ax.grid(True, alpha=0.3)
            else:
                ax.text(0.5, 0.5, 'No valid data', ha='center', va='center', transform=ax.transAxes)
                ax.set_title(f'{condition_name}\n(No valid data)')
        
        # Hide unused subplots
        for idx in range(len(conditions), len(axes)):
            axes[idx].set_visible(False)
        
        # Add overall title and adjust layout
        fig.suptitle(f'Interaction energies for {resnum_chain12} across all conditions{title_suffix}', 
                     fontsize=14, y=0.98)
        plt.tight_layout()
        plt.subplots_adjust(top=0.92)
        plt.show()
    
    # Print comprehensive summary
    print(f"\n=== Summary for residue {residue_id} ===")
    print(f"Total unique interaction pairs: {len(unique_pairs)}")
    print(f"Conditions analyzed: {len(conditions)}")
    
    print(f"\nPairs per condition and allele:")
    for condition_key, allele_counts in peptide_summary.items():
        condition_name = "Peptide-free" if condition_key == 'peptide_free' else f"Peptide: {condition_key}"
        print(f"  {condition_name}: {allele_counts}")
    
    return {
        'filtered_df': df_residue, 
        'unique_pairs': list(unique_pairs), 
        'pair_counts': pair_counts,
        'peptide_summary': peptide_summary,
        'conditions': conditions
    }

In [ ]:
# Analyze residue 74M in peptide-loaded systems
results_74M = analyze_polymorphic_residue_interactions(
    df_cons_pairs, 
    '74M', 
    title_suffix=" (Polymorphic Position)"
)

In [ ]:
# Analyze residue 77M in peptide-loaded systems
results_77M = analyze_polymorphic_residue_interactions(
    df_cons_pairs, 
    '77M', 
    title_suffix=" (Polymorphic Position)"
)

In [ ]:
# Analyze residue 80M in peptide-loaded systems
results_80M = analyze_polymorphic_residue_interactions(
    df_cons_pairs, 
    '80M', 
    title_suffix=" (Polymorphic Position)"
)

In [ ]:
# Analyze residue 81M in peptide-loaded systems
results_81M = analyze_polymorphic_residue_interactions(
    df_cons_pairs, 
    '81M', 
    title_suffix=" (Polymorphic Position)"
)

In [ ]:
# Analyze residue 82M in peptide-loaded systems
results_82M = analyze_polymorphic_residue_interactions(
    df_cons_pairs, 
    '82M', 
    title_suffix=" (Polymorphic Position)"
)

In [ ]:
# Analyze residue 83M in peptide-loaded systems
results_83M = analyze_polymorphic_residue_interactions(
    df_cons_pairs, 
    '83M', 
    title_suffix=" (Polymorphic Position)"
)

In [ ]:
# Analyze residue 95M in peptide-loaded systems
results_95M = analyze_polymorphic_residue_interactions(
    df_cons_pairs, 
    '95M', 
    title_suffix=" (Polymorphic Position)"
)

In [ ]:
# Analyze residue 97M in peptide-loaded systems
results_97M = analyze_polymorphic_residue_interactions(
    df_cons_pairs, 
    '97M', 
    title_suffix=" (Polymorphic Position)"
)